# WildWatch Elmina: train the animal classifier

This notebook trains the AI that recognises **monkey, dog, snake, wild boar, monitor lizard (biawak) and other** animals.
Run the steps in order: click a cell, then press the ▶ play button on its left. Each step says what it does.

**Before you start**
1. **Runtime > Change runtime type > T4 GPU > Save.** Training is about 20× faster on a GPU.
2. Have these two files ready on your computer:
   - `dataset.zip` made by `training/prepare_dataset.py` (6 folders: monkey, dog, snake, wild_boar, monitor_lizard, other)
   - `train.py` from the project's `training/` folder

Total time: about 15 to 30 minutes.

## Step 1: Check the GPU is on

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU is ON:", gpus[0].name, "- good to go.")
else:
    print("No GPU found. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")
    print("(Training still works without a GPU, just much slower.)")

## Step 2: Upload `dataset.zip` and `train.py`

Run the cell, click **Choose files**, and select **both files** at once (hold Ctrl). Wait until the upload reaches 100%.

If `dataset.zip` is very large (over about 500 MB) and the upload is slow, put it in Google Drive instead and use Step 2b.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded))

### Step 2b (only if you used Google Drive instead)
Put `dataset.zip` and `train.py` in a Drive folder called `wildwatch`, then run this cell.

In [ ]:
from google.colab import drive
import shutil
drive.mount("/content/drive")
for name in ["dataset.zip", "train.py"]:
    shutil.copy(f"/content/drive/MyDrive/wildwatch/{name}", f"/content/{name}")
print("Copied from Drive.")

## Step 3: Unpack and check the photos
This shows how many photos each animal has. Aim for at least 150 each.

In [ ]:
import os, shutil, zipfile
from pathlib import Path

os.chdir("/content")
CLASSES = ["monkey", "dog", "snake", "wild_boar", "monitor_lizard", "other"]

assert Path("train.py").exists() or Path("training/train.py").exists(), "train.py not found - upload it in Step 2."
assert Path("dataset.zip").exists(), "dataset.zip not found - upload it in Step 2."
os.makedirs("training", exist_ok=True)
if Path("train.py").exists():
    shutil.move("train.py", "training/train.py")

shutil.rmtree("dataset", ignore_errors=True)
with zipfile.ZipFile("dataset.zip") as z:
    z.extractall("unzipped")
# Find the folder that contains the class folders, wherever it ended up in the zip
root = next((p for p in [Path("unzipped"), *Path("unzipped").rglob("*")]
             if p.is_dir() and sum((p / c).is_dir() for c in CLASSES) >= 2), None)
assert root, "Couldn't find the monkey/dog/snake/... folders inside dataset.zip."
shutil.move(str(root), "dataset")
shutil.rmtree("unzipped", ignore_errors=True)

for c in CLASSES:
    n = len(list(Path("dataset", c).glob("*"))) if Path("dataset", c).exists() else 0
    print(f"{c:<15} {n:>5} photos", "  <- too few, add more" if n < 150 else "")

## Step 4: Train the AI
This is the long step (about 10 to 25 minutes). You'll see each round ("epoch") with its accuracy.
`val_accuracy` is the score on photos the AI has never seen: that's the number that matters.

In [ ]:
!pip install -q scikit-learn matplotlib
!python training/train.py --data dataset --epochs 10 --fine-tune-epochs 5

## Step 5: See the results
Use the accuracy and the confusion matrix in your pitch deck.

In [ ]:
from IPython.display import Image, display
print(open("model/report.txt").read())
display(Image("model/confusion_matrix.png", width=520))

**How to read it**
- **accuracy**: the share of test photos it got right (for example 0.90 = 90%).
- **Confusion matrix**: rows are the real animal, columns are what the AI guessed. A strong diagonal is good. Numbers off the diagonal show which animals it mixes up.
- If one animal scores badly, add more (and more varied) photos of it, rebuild `dataset.zip`, and run again.

## Step 6 (optional): Try it on your own photo

In [ ]:
!pip install -q ai-edge-litert
import json, numpy as np
from google.colab import files
from PIL import Image as PILImage, ImageOps, UnidentifiedImageError
try:
    from ai_edge_litert.interpreter import Interpreter   # newer runtime (same one the app uses)
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter

labels = json.load(open("model/labels.json"))
interp = Interpreter(model_path="model/animal_model.tflite"); interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]

print("Choose one or more animal photos to test:")
for name in files.upload():
    try:
        img = ImageOps.exif_transpose(PILImage.open(name)).convert("RGB")
    except (UnidentifiedImageError, OSError):
        print(f"{name}: not a photo, skipped"); continue
    x = np.expand_dims(np.asarray(img.resize((224, 224)), dtype="float32"), 0)
    interp.set_tensor(inp["index"], x); interp.invoke()
    probs = interp.get_tensor(out["index"])[0]
    display(img.resize((224, int(224 * img.height / img.width))))
    print(name)
    for i in np.argsort(probs)[::-1][:3]:
        print(f"  {labels[i]:<10} {probs[i]*100:5.1f}%")

## Step 7: Download the trained model

In [ ]:
import zipfile
from google.colab import files
# The app only needs the .tflite model and labels; the report and chart are for your slides.
with zipfile.ZipFile("wildwatch_model.zip", "w") as z:
    for f in ["animal_model.tflite", "labels.json", "report.txt", "confusion_matrix.png"]:
        z.write(f"model/{f}", f"model/{f}")
files.download("wildwatch_model.zip")

## Last step: add the AI to the app

1. Unzip `wildwatch_model.zip` on your computer.
2. Copy **`animal_model.tflite`** and **`labels.json`** into the project's `model/` folder.
3. Upload those two files to the GitHub repository (in the `model/` folder).
4. Render redeploys automatically. Open `https://<your-app>.onrender.com/api/health`: it should show `"ai_model": true`.

Keep `confusion_matrix.png` and `report.txt` for the slides. You don't need to upload `animal_model.keras` (it's large and only used for retraining).